In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Parquet").master("local[8]").getOrCreate()
sc = spark.sparkContext

In [26]:
from pyspark.sql.functions import col, split, substring, desc
from pyspark.sql.types import IntegerType, LongType

In [3]:
df_orig = spark.read.options(header=True, inferSchema=True, delimiter="\t").csv("hdfs://hdfs-namenode:9000/task1/input/customs_data.csv")

In [4]:
df_cat = spark.read.options(header=False, inferSchema=False, delimiter="\t").csv("hdfs://hdfs-namenode:9000/task1/output_t1/part-00000")

## Looking at the input data

In [5]:
df_orig.show()
df_orig.printSchema()

+----------+-------+---------+--------+-------+-------+------+--------+------+------+
|      code|country|direction|district|measure|  month| netto|quantity|region| value|
+----------+-------+---------+--------+-------+-------+------+--------+------+------+
|6204695000|     IT|       ИМ|       1|     ШТ|01/2016|     1|       7| 46000|   131|
|9001900009|     CN|       ИМ|       1|      1|01/2016|    18|       0| 46000|112750|
|8414302004|     BY|       ИМ|       6|     ШТ|01/2016|    57|       8| 50000|   392|
|9018509000|     US|       ИМ|       2|      1|01/2016|   179|       0| 40000| 54349|
|9021101000|     EE|       ИМ|       1|      1|01/2016|   372|       0| 46000| 17304|
|3816000000|     FR|       ИМ|       2|      1|01/2016|253600|       0| 40000|323488|
|8523519300|     MX|       ИМ|       2|     ШТ|01/2016|     0|       4| 40000|  1611|
|6204520000|     JP|       ИМ|       1|     ШТ|01/2016|     1|       2| 46000|    29|
|6110209100|     KR|       ИМ|       1|     ШТ|01/2016

In [6]:
df_cat.show(truncate=False)
df_cat.printSchema()

+----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|_c0 |_c1                                                                                                                                                                                        |
+----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|0101|ЛОШАДИ, ОСЛЫ, МУЛЫ И ЛОШАКИ ЖИВЫЕ                                                                                                                                                          |
|0102|КРУПНЫЙ РОГАТЫЙ СКОТ ЖИВОЙ                                                                                                                                                                 |
|0103|СВИНЬИ ЖИВЫЕ       

## Casting some columns, adding meta columns, left joining categories

In [7]:
df_proc = df_orig.withColumn("value", col("value").cast(LongType()))

In [16]:
df_yearMonth = split(df_proc["month"], "/")
df_proc = df_proc.withColumn("meta_year", df_yearMonth.getItem(1).cast(IntegerType())).withColumn("meta_month", df_yearMonth.getItem(0).cast(IntegerType()))
df_proc = df_proc.withColumn("meta_code", substring("code", 1, 4))
df_joined = df_proc.join(
    df_cat,
    df_proc["meta_code"] == df_cat["_c0"],
    "left"
).drop("meta_code", "_c0").fillna("ПРОЧЕЕ", subset="_c1").withColumnRenamed("_c1", "category")

In [17]:
df_joined.show(truncate=True)
df_joined.printSchema()

+----------+-------+---------+--------+-------+-------+------+--------+------+------+---------+----------+--------------------+
|      code|country|direction|district|measure|  month| netto|quantity|region| value|meta_year|meta_month|            category|
+----------+-------+---------+--------+-------+-------+------+--------+------+------+---------+----------+--------------------+
|6204695000|     IT|       ИМ|       1|     ШТ|01/2016|     1|       7| 46000|   131|     2016|         1|КОСТЮМЫ, КОМПЛЕКТ...|
|9001900009|     CN|       ИМ|       1|      1|01/2016|    18|       0| 46000|112750|     2016|         1|ВОЛОКНА ОПТИЧЕСКИ...|
|8414302004|     BY|       ИМ|       6|     ШТ|01/2016|    57|       8| 50000|   392|     2016|         1|НАСОСЫ ВОЗДУШНЫЕ ...|
|9018509000|     US|       ИМ|       2|      1|01/2016|   179|       0| 40000| 54349|     2016|         1|ПРИБОРЫ И УСТРОЙС...|
|9021101000|     EE|       ИМ|       1|      1|01/2016|   372|       0| 46000| 17304|     2016|         

## Write out with selected order

In [18]:
df_selected = df_joined.select("month", "country", "direction", "code", "category", "measure", "value", "netto", "quantity", "region", "district", "meta_year", "meta_month")
df_selected.show()

+-------+-------+---------+----------+--------------------+-------+------+------+--------+------+--------+---------+----------+
|  month|country|direction|      code|            category|measure| value| netto|quantity|region|district|meta_year|meta_month|
+-------+-------+---------+----------+--------------------+-------+------+------+--------+------+--------+---------+----------+
|01/2016|     IT|       ИМ|6204695000|КОСТЮМЫ, КОМПЛЕКТ...|     ШТ|   131|     1|       7| 46000|       1|     2016|         1|
|01/2016|     CN|       ИМ|9001900009|ВОЛОКНА ОПТИЧЕСКИ...|      1|112750|    18|       0| 46000|       1|     2016|         1|
|01/2016|     BY|       ИМ|8414302004|НАСОСЫ ВОЗДУШНЫЕ ...|     ШТ|   392|    57|       8| 50000|       6|     2016|         1|
|01/2016|     US|       ИМ|9018509000|ПРИБОРЫ И УСТРОЙС...|      1| 54349|   179|       0| 40000|       2|     2016|         1|
|01/2016|     EE|       ИМ|9021101000|ПРИСПОСОБЛЕНИЯ ОР...|      1| 17304|   372|       0| 46000|       

In [19]:
df_repartitioned = df_selected.repartition("meta_year", "meta_month")

In [21]:
df_repartitioned.write.mode("overwrite").partitionBy("meta_year", "meta_month").parquet("hdfs://hdfs-namenode:9000/partners")

## Check quality

In [22]:
df_result = spark.read.parquet("hdfs://hdfs-namenode:9000/partners")

In [27]:
df_result.orderBy(col("meta_year").desc()).show()

+-------+-------+---------+----------+--------------------+-------+-----+-----+--------+------+--------+---------+----------+
|  month|country|direction|      code|            category|measure|value|netto|quantity|region|district|meta_year|meta_month|
+-------+-------+---------+----------+--------------------+-------+-----+-----+--------+------+--------+---------+----------+
|01/2021|     CN|       ИМ|9403301900|МЕБЕЛЬ ПРОЧАЯ И Е...|     ШТ| NULL|    0|     105|  5000|       7|     2021|         1|
|03/2021|     KR|       ИМ|9031803800|ИЗМЕРИТЕЛЬНЫЕ ИЛИ...|     ШТ| NULL|   20|      16|  5000|       7|     2021|         3|
|09/2021|     DE|       ИМ|8483908909|ВАЛЫ ТРАНСМИССИОН...|      1| NULL|    0|       0|  5000|       7|     2021|         9|
|03/2021|     UA|       ИМ|8205510090|ИНСТРУМЕНТЫ РУЧНЫ...|      1| NULL|    0|       0| 60000|       3|     2021|         3|
|02/2021|     RO|       ЭК|8481801900|КРАНЫ, КЛАПАНЫ, В...|      1| NULL|    0|       0| 40000|       2|     2021|    

In [28]:
df_result.printSchema()

root
 |-- month: string (nullable = true)
 |-- country: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- code: string (nullable = true)
 |-- category: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- value: long (nullable = true)
 |-- netto: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- region: integer (nullable = true)
 |-- district: integer (nullable = true)
 |-- meta_year: integer (nullable = true)
 |-- meta_month: integer (nullable = true)

